In [6]:
%load_ext autoreload
%autoreload 2

import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.optim import Adam,AdamW

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt

from utils import simdatset, reproducibility

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/disk1/user/liaoshuilin/project/35.TAPE_EXO/assay_diffusion


# Load data

In [7]:
import pickle

with open(f'../result/data_abalation/Stim_data.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

GTE_x_train = loaded_data['GTE_x_train']
GTE_x_val = loaded_data['GTE_x_val']
GTE_x_test = loaded_data['GTE_x_test']
GTE_y_train = loaded_data['GTE_y_train']
GTE_y_val = loaded_data['GTE_y_val']
GTE_y_test = loaded_data['GTE_y_test']

HPA_x = loaded_data['HPA_x']
HPA_y = loaded_data['HPA_y']
real_x = loaded_data['real_x']

all_genename = loaded_data['all_genename']
celltypes = loaded_data['celltypes']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_pth = "../result/model_abalation/"

In [5]:
genename_df = pd.DataFrame(all_genename, columns=["GeneName"])
genename_df.to_csv(out_pth + 'genename.csv', index=False)

celltypes_df = pd.DataFrame(celltypes, columns=["TissueType"])
celltypes_df.to_csv(out_pth + 'TissueType.csv', index=False)

pd.DataFrame(HPA_y).to_csv(out_pth + 'HPA_y_real.csv', index=False, header=False)

In [ ]:
pd.DataFrame(HPA_y).to_csv(out_pth + 'HPA_y_real.csv', index=False, header=False)

In [3]:
print(type(all_genename))
print(type(celltypes))

<class 'list'>
<class 'pandas.core.indexes.base.Index'>


# Stage1 Model

In [6]:
%load_ext autoreload
%autoreload 2
from train_re import AdaptiveTAPEandDiffusion2, alternate_training_earlyStop
from utils import reproducibility

batch_size = 256
reproducibility(2025)
model = AdaptiveTAPEandDiffusion2(GTE_x_train.shape[1], GTE_y_train.shape[1],2, T=2000).to(device)

optimizer_main = AdamW([
    {'params': model.encoder.parameters()},
    {'params': model.predictor.parameters()},
    {'params': model.decoder.parameters()}], lr=1e-4)
optimizer_diffusion =AdamW(model.ref_creator.parameters(), lr=1e-3)
optimizer_all = AdamW(model.parameters(),1e-4)
epochs_main = 400 
epochs_diffusion = 1500

train_loader = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(simdatset(GTE_x_val, GTE_y_val), batch_size=batch_size, shuffle=False)

model, main_loss, diffloss = alternate_training_earlyStop(model, train_loader, val_loader, optimizer_main, optimizer_diffusion, 
                                                 epochs_main, epochs_diffusion, device='cpu',
                                                 patience=5, early_stop_start=400, early_stop_interval=50)
torch.save(model, out_pth + "model_stage1.pth")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Epoch [1/400], Main Loss: 0.2435
Epoch [2/400], Main Loss: 0.0515
Epoch [3/400], Main Loss: 0.0187
Epoch [4/400], Main Loss: 0.0144
Epoch [5/400], Main Loss: 0.0126
Epoch [6/400], Main Loss: 0.0117
Epoch [7/400], Main Loss: 0.0109
Epoch [8/400], Main Loss: 0.0105
Epoch [9/400], Main Loss: 0.0100
Epoch [10/400], Main Loss: 0.0096
Epoch [11/400], Main Loss: 0.0093
Epoch [12/400], Main Loss: 0.0089
Epoch [13/400], Main Loss: 0.0088
Epoch [14/400], Main Loss: 0.0084
Epoch [15/400], Main Loss: 0.0082
Epoch [16/400], Main Loss: 0.0080
Epoch [17/400], Main Loss: 0.0078
Epoch [18/400], Main Loss: 0.0076
Epoch [19/400], Main Loss: 0.0075
Epoch [20/400], Main Loss: 0.0073
Epoch [21/400], Main Loss: 0.0071
Epoch [22/400], Main Loss: 0.0070
Epoch [23/400], Main Loss: 0.0069
Epoch [24/400], Main Loss: 0.0068
Epoch [25/400], Main Loss: 0.0066
Epoch [26/400], Main Loss: 0.0066
Epoch [27/400], Main Loss: 0.0065
Epo

# Get sigmatrix

In [18]:
%load_ext autoreload
%autoreload 2
from train_re import evaluation
from utils import calculate_evaluation_metrics

model = torch.load(out_pth + "model_stage1.pth", weights_only=False, map_location=device)
train_loader2 = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=False)
x_recon_tr, f_tr, z_tr  = evaluation(train_loader2, model, device=device)

sigmatrix = np.linalg.pinv(f_tr) @ x_recon_tr 
pd.DataFrame(sigmatrix).to_csv(out_pth + 'sigmatrix.csv', index=False, header=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# GTEs, SIG

In [9]:
%load_ext autoreload
%autoreload 2
from utils import realdatset
from train_re import get_frac_from_sigmatrix

test_loader =DataLoader(realdatset(GTE_x_test), batch_size=GTE_x_test.shape[0], shuffle=False)

model.eval()
model.state = 'test'

for _, X in enumerate(test_loader):
    X = X.to(device)
    z = model.encode(X).detach().cpu()
    pred_f = get_frac_from_sigmatrix(model, z, sigmatrix).numpy()
    # x_recon = model.decode(z).detach().cpu()
    x_recon = pred_f @ sigmatrix

pd.DataFrame(x_recon).to_csv(out_pth + 'GTE_SIG_x_recon.csv', index=False, header=False)
pd.DataFrame(pred_f).to_csv(out_pth + 'GTE_SIG_y.csv', index=False, header=False)

x_recon = pd.read_csv(out_pth + 'GTE_SIG_x_recon.csv', header=None)
pred_f =  pd.read_csv(out_pth + 'GTE_SIG_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon, input_X = GTE_x_test, out_pth = out_pth + "GTE_SIG_x_")
print(f"RMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = pred_f, input_X = GTE_y_test, out_pth = out_pth + "GTE_SIG_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
RMSE of x: 0.0221
PCC of x: 0.9931
MAE of x: 0.0145
CCC of x: 0.9893
RMSE of test frac: 0.0186
PCC of frac: 0.9246
MAE of frac: 0.0127
CCC of frac: 0.9130


# GTE, Stage1AE

In [8]:
%load_ext autoreload
%autoreload 2
from train_re import evaluation
from utils import calculate_evaluation_metrics

model = torch.load(out_pth + "model_stage1.pth", weights_only=False, map_location=device)
test_loader = DataLoader(simdatset(GTE_x_test, GTE_y_test), batch_size=batch_size, shuffle=False)
x_recon_te, f_te, z_te  = evaluation(test_loader, model, device=device)
pd.DataFrame(x_recon_te).to_csv(out_pth + 'GTE_AE_x_recon.csv', index=False, header=False)
pd.DataFrame(f_te).to_csv(out_pth + 'GTE_AE_y.csv', index=False, header=False)

x_recon_te = pd.read_csv(out_pth + 'GTE_AE_x_recon.csv', header=None)
f_te =  pd.read_csv(out_pth + 'GTE_AE_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_te, input_X = GTE_x_test, out_pth = out_pth + "GTE_AE_x_")
print(f"RMSE of test x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_te, input_X = GTE_y_test, out_pth = out_pth + "GTE_AE_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
RMSE of test x: 0.0147
PCC of x: 0.9962
MAE of x: 0.0094
CCC of x: 0.9954
RMSE of test frac: 0.0102
PCC of frac: 0.9852
MAE of frac: 0.0071
CCC of frac: 0.9703


# HPA，Stage1AE

In [4]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain5
from utils import calculate_evaluation_metrics

x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain5(x=HPA_x, model_name= out_pth + "model_stage1",  adaptive=False, device=device)
pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'HPA_AE_x_recon.csv', index=False, header=False)
pd.DataFrame(f_HPA).to_csv(out_pth + 'HPA_AE_y.csv', index=False, header=False)

x_recon_HPA = pd.read_csv(out_pth + 'HPA_AE_x_recon.csv', header=None)
f_HPA =  pd.read_csv(out_pth + 'HPA_AE_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_AE_x_")
print(f"RMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_AE_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
RMSE of x: 0.0721
PCC of x: 0.9000
MAE of x: 0.0509
CCC of x: 0.8912
RMSE of test frac: 0.0417
PCC of frac: 0.7562
MAE of frac: 0.0275
CCC of frac: 0.7363


# HPA，SIG

In [7]:
from utils import realdatset
from train_re import get_frac_from_sigmatrix
model.eval()
model.state = 'test'

test_loader =DataLoader(realdatset(HPA_x), batch_size=HPA_x.shape[0], shuffle=False)

for _, X in enumerate(test_loader):
    X = X.to(device)
    z = model.encode(X).detach().cpu()
    pred_f = get_frac_from_sigmatrix(model, z, sigmatrix).numpy()
    # x_recon = model.decode(z).detach().cpu()
    x_recon = pred_f @ sigmatrix

pd.DataFrame(x_recon).to_csv(out_pth + 'HPA_SIG_x_recon.csv', index=False, header=False)
pd.DataFrame(pred_f).to_csv(out_pth + 'HPA_SIG_y.csv', index=False, header=False)

x_recon = pd.read_csv(out_pth + 'HPA_SIG_x_recon.csv', header=None)
pred_f =  pd.read_csv(out_pth + 'HPA_SIG_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon, input_X = HPA_x, out_pth = out_pth + "HPA_SIG_x_")
print(f"RMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = pred_f, input_X = HPA_y, out_pth = out_pth + "HPA_SIG_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

RMSE of x: 0.0838
PCC of x: 0.8950
MAE of x: 0.0623
CCC of x: 0.8661
RMSE of test frac: 0.0373
PCC of frac: 0.7572
MAE of frac: 0.0255
CCC of frac: 0.7370


# HPA, DADA noDF

In [8]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain9_noDF
from utils import calculate_evaluation_metrics

x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain9_noDF(x=HPA_x, sour_x_train = GTE_x_train, model_name=out_pth + "model_stage1", mode = 'overall5', steps=10, max_iter=40, device=device, sigmatrix = sigmatrix)
pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'HPA_DADA_noDF_x_recon.csv', index=False, header=False)
pd.DataFrame(f_HPA).to_csv(out_pth + 'HPA_DADA_noDF_y.csv', index=False, header=False)
torch.save(model2, out_pth + "model_stage2_noDF.pth")

x_recon_HPA = pd.read_csv(out_pth + 'HPA_DADA_noDF_x_recon.csv', header=None)
f_HPA =  pd.read_csv(out_pth + 'HPA_DADA_noDF_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_DADA_noDF_x_")
print(f"RMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_DADA_noDF_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
RMSE of x: 0.0261
PCC of x: 0.9902
MAE of x: 0.0183
CCC of x: 0.9858
RMSE of test frac: 0.0260
PCC of frac: 0.8727
MAE of frac: 0.0175
CCC of frac: 0.8553


# HPA，DADA-EV

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain9
reproducibility(2025)
# sigmatrix =  pd.read_csv(out_pth + 'sigmatrix.csv')
x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain9(x=HPA_x, model_name=out_pth + "model_stage1", mode = 'overall5', steps=10, max_iter=40, device=device, sigmatrix = sigmatrix)
pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'HPA_DADA_x_recon.csv', index=False, header=False)
pd.DataFrame(f_HPA).to_csv(out_pth + 'HPA_DADA_y.csv', index=False, header=False)

torch.save(model2.state_dict(), out_pth + "model_stage2_dict.pth")
torch.save(model2, out_pth + "model_stage2.pth")

In [7]:
%load_ext autoreload
%autoreload 2
from utils import calculate_evaluation_metrics

x_recon_HPA = pd.read_csv(out_pth + 'HPA_DADA_x_recon.csv', header=None)
f_HPA =  pd.read_csv(out_pth + 'HPA_DADA_y.csv', header=None)
print(x_recon_HPA.shape)
print(f_HPA.shape)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_DADA_x_")
print(f"RMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_DADA_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(50, 18757)
(50, 23)
RMSE of x: 0.0253
PCC of x: 0.9902
MAE of x: 0.0173
CCC of x: 0.9866
RMSE of test frac: 0.0269
PCC of frac: 0.8539
MAE of frac: 0.0184
CCC of frac: 0.8325


# embedding compare

In [3]:
import torch
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# 随机种子
np.random.seed(2025)
torch.manual_seed(2025)

# 假设 X_source, X_target 都是 tensor 或 ndarray
X_source = torch.tensor(GTE_x_train).to(device)
X_target = torch.tensor(HPA_x).to(device)
model = torch.load(out_pth + "model_stage1.pth", weights_only=False, map_location=device) 
model2 = torch.load(out_pth + "model_stage2.pth", weights_only=False, map_location=device) 

n = min(len(X_source), len(X_target), 1000)
idx_source = torch.from_numpy(np.random.choice(len(X_source), n, replace=False)).to(device)
idx_target = torch.from_numpy(np.random.choice(len(X_target), n, replace=False)).to(device)

X_source_sample = X_source[idx_source].float()
X_target_sample = X_target[idx_target].float()

with torch.no_grad():
    z_source_ori = model.encode(X_source_sample).cpu().numpy()
    z_target_ori = model.encode(X_target_sample).cpu().numpy()
    z_source = model2.encode(X_source_sample).cpu().numpy()
    z_target = model2.encode(X_target_sample).cpu().numpy()

# 合并4组
X_concat = np.vstack([z_source_ori, z_target_ori, z_source, z_target])
labels = np.array([0]*n + [1]*n + [2]*n + [3]*n)

# t-SNE降维
tsne = TSNE(n_components=2, random_state=42)
z_embedded = tsne.fit_transform(X_concat)

z_embedded_label = np.column_stack((labels, z_embedded))
print(z_embedded_label.shape) 
pd.DataFrame(z_embedded_label).to_csv(out_pth + 'embedding.csv', index=True, header=True)

z_embedded_label = np.column_stack((labels, z_embedded))
print(z_embedded_label.shape) 
pd.DataFrame(z_embedded_label).to_csv(out_pth + 'embedding.csv', index=True, header=True)